In [1]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import VarianceThreshold





In [2]:
covertype = pd.read_csv("../data/covtype.csv")
covertype.head()


,Elevation,Aspect,Slope,Horizontal_Distance_To_Hydrology,Vertical_Distance_To_Hydrology,Horizontal_Distance_To_Roadways,Hillshade_9am,Hillshade_Noon,Hillshade_3pm,Horizontal_Distance_To_Fire_Points,...,Soil_Type32,Soil_Type33,Soil_Type34,Soil_Type35,Soil_Type36,Soil_Type37,Soil_Type38,Soil_Type39,Soil_Type40,Cover_Type
0,2596,51,3,258,0,510,221,232,148,6279,...,0,0,0,0,0,0,0,0,0,5
1,2590,56,2,212,-6,390,220,235,151,6225,...,0,0,0,0,0,0,0,0,0,5
2,2804,139,9,268,65,3180,234,238,135,6121,...,0,0,0,0,0,0,0,0,0,2
3,2785,155,18,242,118,3090,238,238,122,6211,...,0,0,0,0,0,0,0,0,0,2
4,2595,45,2,153,-1,391,220,234,150,6172,...,0,0,0,0,0,0,0,0,0,5


Create X y split

In [3]:
X = covertype.drop('Cover_Type', axis=1)
y = covertype['Cover_Type']


ANOTHER NOTE: 10/14/2026 written before the paragraph below this. I think that the webpage for the assignment 6 and 7 homework is different than the webpage for the assignment found in the "content" tab. I believe it is also in a slightly different order. i just noticed that the homework tab includes some stuff about the ames.csv dataset but the webpage found from the description in the actual assignment does not include ames.csv. So i will conclude that the webpage is slightly different in order and also content. so I am not going to delete what i had added about ames.csv (which is simply just loading the dataset). I hope this is not too confusing. it was pretty confusing for me.  

varience threshold 
i realize now (10/14/2026) that you ask this later. I address this later in the <br>
notebook that i think i accidentally moved ahead because I was experiencing bad headaches <br>
and consequently some probable brain fog due to hayfever at the beginning of this week which <br>
was really interfering with my day to day tasks. i did not want to rearrange my code though <br>
because when i calculate the conditioning number i test it on the variance threshold affected data<br>
as well as the data that had not been affected by my varience threshold

In [4]:
#first we will do it the "incorrect way" by creating our variance threshold before cross validation. both will have the threshold of < 5%
selector = VarianceThreshold(threshold=0.05)
X_preCV = selector.fit_transform(X)




## About your question of collinearity
Collinearity is when 2 or more parameters are highly correlated.
Models that are sensitive to collinearity: linear regression, logistic regression, naive bayes, MLP
Distance based models like KNN, K-Means and SVMs can have distorted metrics by collinearity.

In [5]:
corr_matrix = X.corr(method='pearson')
#commented out because there is too much blocking the screen
print(corr_matrix)


                                    Elevation    Aspect     Slope  \
Elevation                            1.000000  0.015735 -0.242697   
Aspect                               0.015735  1.000000  0.078728   
Slope                               -0.242697  0.078728  1.000000   
Horizontal_Distance_To_Hydrology     0.306229  0.017376 -0.010607   
Vertical_Distance_To_Hydrology       0.093306  0.070305  0.274976   
Horizontal_Distance_To_Roadways      0.365559  0.025121 -0.215914   
Hillshade_9am                        0.112179 -0.579273 -0.327199   
Hillshade_Noon                       0.205887  0.336103 -0.526911   
Hillshade_3pm                        0.059148  0.646944 -0.175854   
Horizontal_Distance_To_Fire_Points   0.148022 -0.109172 -0.185662   
Wilderness_Area1                     0.131838 -0.140123 -0.234576   
Wilderness_Area2                     0.238164  0.055988 -0.036253   
Wilderness_Area3                     0.066550  0.074904  0.125663   
Wilderness_Area4                  

i have a nice setup in the cell below that makes it easier to see whats correlated

In [6]:
threshold = 0.8

mask = (corr_matrix.abs() > threshold) & (corr_matrix != 1.0) # everything that is below the threshold or equals 1 (e.g., elevation == elevation equals 1.0) is labelled false. it lets me view the pearson chart more efficiently

print(mask)

                                    Elevation  Aspect  Slope  \
Elevation                               False   False  False   
Aspect                                  False   False  False   
Slope                                   False   False  False   
Horizontal_Distance_To_Hydrology        False   False  False   
Vertical_Distance_To_Hydrology          False   False  False   
Horizontal_Distance_To_Roadways         False   False  False   
Hillshade_9am                           False   False  False   
Hillshade_Noon                          False   False  False   
Hillshade_3pm                           False   False  False   
Horizontal_Distance_To_Fire_Points      False   False  False   
Wilderness_Area1                        False   False  False   
Wilderness_Area2                        False   False  False   
Wilderness_Area3                        False   False  False   
Wilderness_Area4                        False   False  False   
Soil_Type1                              

## Summarization of the cell directly below
So we can see there are values that have corellation above abs(50%). 
As you state in the homework, negative corellation is still a corellation.

In [7]:
#another vibe coded piece. lets see what it does? 
#note: i added the & (corr_matrix != 1.0) to set situations like Elevation==Elevation equal to NaN instead of 1.0
threshold = 0.5

high_corr = corr_matrix.where((corr_matrix.abs() > threshold) & (corr_matrix != 1.0) ) 
print(high_corr)
#interesting. it shows NaN instead of false, which makes it easier to read 
#shows that there are multiple categories with >abs(50%) correlation

                                    Elevation    Aspect     Slope  \
Elevation                                 NaN       NaN       NaN   
Aspect                                    NaN       NaN       NaN   
Slope                                     NaN       NaN       NaN   
Horizontal_Distance_To_Hydrology          NaN       NaN       NaN   
Vertical_Distance_To_Hydrology            NaN       NaN       NaN   
Horizontal_Distance_To_Roadways           NaN       NaN       NaN   
Hillshade_9am                             NaN -0.579273       NaN   
Hillshade_Noon                            NaN       NaN -0.526911   
Hillshade_3pm                             NaN  0.646944       NaN   
Horizontal_Distance_To_Fire_Points        NaN       NaN       NaN   
Wilderness_Area1                          NaN       NaN       NaN   
Wilderness_Area2                          NaN       NaN       NaN   
Wilderness_Area3                          NaN       NaN       NaN   
Wilderness_Area4                  

## find conditioning number

In [8]:
#preprocess first before finding conditioning number
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
cond_X = np.linalg.cond(X_scaled)   
print(cond_X)
# preprocess (scale) on varianced dataset
cond_X_preCV_SCALED = scaler.fit_transform(X_preCV)
cond_X_preCV_SCALED_COND = np.linalg.cond(cond_X_preCV_SCALED) 
print(cond_X_preCV_SCALED_COND)

1206928151499887.0
31.966985026973195


## thoughts on the previous cell
removing the unneccesary columns (soil) that were included before using a variance threshold significantly reduced the conditioning number of the dataset, meaning that post variance manipulation reduced the conditioning number to the point that it signifies only mild collinearity. this is big for logistic regression, naive, and linear models that cannot handle that type of data.

## low variance predictors
i confirm that all of the soil type columns are one hot

In [9]:

unique_values = X[['Soil_Type1', 'Soil_Type2', 'Soil_Type3', 'Soil_Type4', 'Soil_Type5', 'Soil_Type6', 'Soil_Type7', 'Soil_Type8', 'Soil_Type9', 'Soil_Type10', 'Soil_Type11', 'Soil_Type12', 'Soil_Type13', 'Soil_Type14', 'Soil_Type15', 'Soil_Type16', 'Soil_Type17', 'Soil_Type18', 'Soil_Type19', 'Soil_Type20', 'Soil_Type21', 'Soil_Type22', 'Soil_Type23', 'Soil_Type24', 'Soil_Type25', 'Soil_Type26', 'Soil_Type27', 'Soil_Type28', 'Soil_Type29', 'Soil_Type30', 'Soil_Type31', 'Soil_Type32', 'Soil_Type33', 'Soil_Type34', 'Soil_Type35', 'Soil_Type36', 'Soil_Type37', 'Soil_Type38', 'Soil_Type39', 'Soil_Type40']].nunique()
non_unique = [val for val in unique_values if val != 2]
# we make sure that soiltype is one hot, which it is
print(non_unique)

[]


## about variance threshold
so i did the variance threshold around the beginning of this jupyter notebook.<br>i did not want to rearrange so much in this notebook (i had confirmed the one hot in the cell directly above this a few days ago and had to rearrange it and put the code down here to align with the order of the homework) <br>

I think the reason i did this is that I had been dealing with hayfever since the begining of this week so I was getting realy bad headaches and it was messing with my head so I had a hard time keeping track of where i was. <br>
so the variance threshold is in one of the uppoer cells near the beginning and i dont want to change the order because it will affect my conditioning number calculation, which i thought was substantial toward the project, even if it's not something you asked for.<br>

i set the variance threshold to 5% and there were a few categories that indicated collinearity greater than abs(50%) <br>
also removing the categories that applied to my very low varience threshold significantly affected <br>
the conditioning number scores significantly. <br>

## find amount of columns removed due to low varience (<5%)

In [10]:
# 1. Get kept columns
kept_columns = selector.get_feature_names_out(X.columns)

# 2. Find removed columns
removed_columns = set(X.columns) - set(kept_columns)

print("Removed columns:", list(removed_columns))
print("num of columns removed: ",len(list(removed_columns)))
print("number of columns kept: ",len(set(kept_columns)))
#significantly lowered the number of columns 
print("number of columns originally", len(set(X.columns)))
print("we have significatly reduced the number of columns due to low varience. we see later that it does in fact affect CV score.")

Removed columns: ['Soil_Type19', 'Soil_Type11', 'Soil_Type24', 'Wilderness_Area2', 'Soil_Type36', 'Soil_Type15', 'Soil_Type21', 'Soil_Type18', 'Soil_Type25', 'Soil_Type3', 'Soil_Type6', 'Soil_Type8', 'Soil_Type38', 'Soil_Type20', 'Soil_Type35', 'Soil_Type9', 'Soil_Type26', 'Soil_Type16', 'Soil_Type1', 'Soil_Type7', 'Soil_Type5', 'Soil_Type4', 'Soil_Type28', 'Soil_Type17', 'Soil_Type40', 'Soil_Type2', 'Soil_Type12', 'Soil_Type14', 'Soil_Type39', 'Soil_Type27', 'Soil_Type13', 'Soil_Type31', 'Soil_Type37', 'Soil_Type34', 'Soil_Type30']
num of columns removed:  35
number of columns kept:  19
number of columns originally 54
we have significatly reduced the number of columns due to low varience. we see later that it does in fact affect CV score.


In [11]:
#do it again a different way just to make sure
# selector.get_support() returns [True, False, True, True, ...]
mask = selector.get_support()

# True counts as 1, False counts as 0
num_kept = mask.sum()
num_removed = (~mask).sum()
print("number of columns kept", num_kept)
print("number of columns removed", num_removed)
print("total columns originally",len(X.columns))

number of columns kept 19
number of columns removed 35
total columns originally 54


## finally we will start fitting our models with varied and unvaried data

In [12]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
lda_pipe = make_pipeline(StandardScaler(), LinearDiscriminantAnalysis())
qda_pipe = make_pipeline(StandardScaler(), QuadraticDiscriminantAnalysis(reg_param=0.1))
nb_pipe  = make_pipeline(StandardScaler(), GaussianNB())

# Cross-validation with pipeline
lda_cv = cross_val_score(lda_pipe, X_train, y_train, cv=5)
qda_cv = cross_val_score(qda_pipe, X_train, y_train, cv=5)
nb_cv  = cross_val_score(nb_pipe, X_train, y_train, cv=5)

# varience threshold affected data <5% varience columns removed
X_train, X_test, y_train, y_test = train_test_split(X_preCV, y, test_size=0.2, random_state=42)
lda_cv1 = cross_val_score(lda_pipe, X_train, y_train, cv=5)
qda_cv1 = cross_val_score(qda_pipe, X_train, y_train, cv=5)
nb_cv1  = cross_val_score(nb_pipe, X_train, y_train, cv=5)


create SVC pipeline to compare
<br> scrapped because of time constraint

In [ ]:
#add them down here so i dont need to reload my models

print("LDA Mean CV:", lda_cv.mean())
print("QDA Mean CV:", qda_cv.mean())
print("NB Mean CV: ", nb_cv.mean())
print("LDA Mean CV with varience:", lda_cv1.mean())
print("QDA Mean CV with varience:", qda_cv1.mean())
print("NB Mean CV with varience: ", nb_cv1.mean())



LDA Mean CV: 0.6796576660994655
QDA Mean CV: 0.4429884101626026
NB Mean CV:  0.08751766765138955
LDA Mean CV with varience: 0.6752257397430632
QDA Mean CV with varience: 0.5842916125005141
NB Mean CV with varience:  0.19722509700914426


# in conclusion 
Linear Discriminant Analysis performed the best overall. Varience made it perform worse. LDA was negatively impacted by removing columns in varience. The reason LDA performed so well is because it assumes every class shares a covarience matrix and draws linear decision boundaries. 
<br>
Quadratic Discriminant Analysis estimates a separate covariance matrix for each class, which can lead to overfitting and overlap if there are collinear features. This also explains why removing low varience classes improves QDA. <br>
Naive Bayes performed the worst overall. NB assumes all classes are independent. With collinear features, which our dataset did indeed have, NB uses Bayes theorem, multiplying every class probability, causing the collinear classes to have multiplied their influence, leading to overfitting. This also explains why removing the low varience features improved our NB model.

### about SVC
I did not run SVC because it took forever. The dataset is huge and I need to submit this homework so i can move on. I have been so sick with hayfever that I missed working most of this week and i am behind.